In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/2_PerenAI_dataset_clean.csv")

# Calcul du Body Age Score

In [2]:
# activité physique 
activity_score_map_body_age = {
    "0–1": 3,
    "0": 3,
    "1": 3,
    "2–3": 1,
    "2": 1,
    "3": 1,
    "4": 0,
    "4–5": 0,
    "5": 0,
    "≥6": -2
}

#BMI(IMC)
def bmi_score(bmi):
    if bmi < 18.5:
        return 1
    elif 18.5 <= bmi < 25:
        return 0
    elif 25 <= bmi < 30:
        return 1
    else:
        return 3


# Sleep_duration
sleep_score_map_body_age = {
    "<6h": 1,
    "6–7h": 0,
    "7–8h": 0,
    ">8h": -1
}

#Stress
stress_score_map_body_age = {
    "high": 2,
    "moderate": 1,
    "low": 0
}

#Alimentation
nutrition_score_map_body_age = {
    "poor": 2,
    "mixed": 0,
    "equilibrated": -1
}



In [3]:
df = df.copy()

df["activity_score"] = df["activity_freq"].map(activity_score_map_body_age).fillna(0)
df["bmi_score"] = df["bmi"].apply(bmi_score)
df["sleep_score"] = df["sleep_duration"].map(sleep_score_map_body_age).fillna(0)
df["stress_score"] = df["stress_level_norm"].map(stress_score_map_body_age).fillna(0)
df["nutrition_score"] = df["nutrition_norm"].map(nutrition_score_map_body_age).fillna(0)

df["body_age"] = (
    df["age"]
    + df["activity_score"]
    + df["bmi_score"]
    + df["sleep_score"]
    + df["stress_score"]
    + df["nutrition_score"]
)



# Calcul du Work load Score

In [4]:
# frequence sportive
activity_freq_map_workload = {
    "0–1": 0,
    "0": 0,
    "1": 0,
    "2–3": 10,
    "2": 10,
    "3": 10,
    "4": 20,
    "4–5": 20,
    "5": 20,
    "≥6": 30
}
#Stress 
stress_map_workload = {
    "low": 0,
    "moderate": -10,
    "high": -20,
    "very_high": -30
}




In [5]:
# --- Work Load Score ---

# 1. Intensité sportive (non disponible → neutre)
df["intensity_score_workload"] = 0

# 2. Fréquence sportive
df["activity_score_workload"] = df["activity_freq"].map(activity_freq_map_workload).fillna(0)

# 3. Stress
df["stress_score_workload"] = df["stress_level_norm"].map(stress_map_workload).fillna(0)

# 4. Dette de sommeil
df["sleep_debt_score_workload"] = df["sleep_6h_plus_norm"].apply(
    lambda x: 0 if x == 1 else 20
)

# 5. Calcul final
df["work_load"] = (
    df["intensity_score_workload"]
    + df["activity_score_workload"]
    - df["stress_score_workload"]
    - df["sleep_debt_score_workload"]
)


# Calcul du Body toxins score

In [6]:
# Nutrition
nutrition_exposure_map_toxins = {
    "ultra_processed": 2,
    "mixed": 0,
    "equilibrated": 0
}

# Alcool
alcohol_exposure_map_toxins = {
    "Jamais": 0,
    "Occasionnel": 0,
    "Régulier": 2,
    "1–3/sem": 2,
    ">3/sem": 2
}


In [7]:
# Hydratation (non disponible → neutre)
df["Hydratation_score_toxins"] = 0
df["nutrition_toxin"] = df["nutrition_norm"].map(nutrition_exposure_map_toxins).fillna(0)
df["alcohol_toxin"] = df["alcohol_raw"].map(alcohol_exposure_map_toxins).fillna(0)

df["sport_toxins"] = df["activity_freq"].apply(
    lambda x: -2 if x in ["3", "4", "4–5", "≥5"] else 0

)

df["Body_Toxins"] = (
    df["nutrition_toxin"]
    + df["alcohol_toxin"]
    - (df["Hydratation_score_toxins"] + df["sport_toxins"])
)


In [8]:
final_features = df[[
    "user_id",
    "datetime",
    "sex",
    "age",
    "height_cm",
    "weight_kg"
]].copy()

final_features["body_age"] = df["body_age"]
final_features["work_load"] = df ["work_load"]
final_features["body_toxin"] = df["Body_Toxins"]
final_features["sync_cycle"] = None
final_features["energy"] = None
final_features["recovery"] = None


In [9]:
long_df = final_features.drop(columns=["sync_cycle", "energy", "recovery"])


In [10]:
final_features.to_csv(
    "../data/processed/3_PerenAI_features_v1.csv",
    index=False
)


In [11]:
long_df.to_csv(
    "../data/processed/PerenAI_3scores.csv",
    index=False
)


In [12]:
long_df.head()

,user_id,datetime,sex,age,height_cm,weight_kg,body_age,work_load,body_toxin
0,U01,2025-10-18 08:12:34,female,32,165,58,32.0,10.0,2.0
1,U01,2025-11-02 07:58:21,female,32,165,59,32.0,10.0,2.0
2,U01,2025-11-15 08:20:41,female,32,165,58,32.0,10.0,2.0
3,U02,2025-10-20 10:45:12,male,38,178,75,40.0,40.0,4.0
4,U02,2025-11-04 09:40:08,male,38,178,76,39.0,30.0,4.0
